In [6]:
!pip install -U \
openai \
langchain[openai] \
langchain-openai \
langchain-community \
chromadb \
gradio \
python-dotenv \
sentence-transformers \
langchain-text-splitters \
requests

zsh:1: no matches found: langchain[openai]


In [7]:
from dotenv import load_dotenv
import os

load_dotenv()
os.environ["OPENAI_API_KEY"]

KeyError: 'OPENAI_API_KEY'

In [9]:
#SERVICE 1

import requests
from openai import OpenAI

client = OpenAI()

def get_weather(city: str):

    geo_url = f"https://geocoding-api.open-meteo.com/v1/search?name={city}&count=1"
    geo_resp = requests.get(geo_url).json()
    
    if "results" not in geo_resp:
        return "I couldn't find that location."

    lat = geo_resp["results"][0]["latitude"]
    lon = geo_resp["results"][0]["longitude"]

    weather_url = (
        f"https://api.open-meteo.com/v1/forecast?"
        f"latitude={lat}&longitude={lon}&current_weather=true"
    )
    weather_data = requests.get(weather_url).json()
    current = weather_data["current_weather"]

    prompt = f"""
    Convert wether data into a friendly summary:

    Temperature: {current['temperature']}°C
    Wind Speed: {current['windspeed']} km/h
    Weather Code: {current['weathercode']}
    """

    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[{"role": "user", "content": prompt}]
    )

    return response.choices[0].message.content

OpenAIError: The api_key client option must be set either by passing api_key to the client or by setting the OPENAI_API_KEY environment variable

In [ ]:
# SERVICE 2 SEMANTIC QUEY

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain.vectorstores import Chroma

# Load text
with open("data.txt") as f:
    text = f.read()

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
docs = splitter.create_documents([text])

embeddings = OpenAIEmbeddings()

vectorstore = Chroma.from_documents(
    docs,
    embedding=embeddings,
    persist_directory="./chroma_db"
)

vectorstore.persist()

In [ ]:
# GUARD RAIL
RESTRICTED_TOPICS = [
    "cat", "dog",
    "horoscope", "zodiac",
    "taylor swift"
]

def guardrails(user_input):
    lowered = user_input.lower()

    if any(word in lowered for word in RESTRICTED_TOPICS):
        return "I cannot discuss that topic."

    if "system prompt" in lowered:
        return "I cannot reveal this."
